# Regime Model Pipeline Orchestration v01

Thin orchestration runner for dataset, training, and module-based static optimization stages.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

NOTEBOOK_UTILS_SRC = PROJECT_ROOT / 'notebook_utils' / 'src'
if str(NOTEBOOK_UTILS_SRC) not in sys.path:
    sys.path.append(str(NOTEBOOK_UTILS_SRC))

from models.regime_model import (
    RunConfig,
    StaticOptimizationConfig,
    build_feature_dataset,
    run_full_pipeline,
    run_static_optimization_stage,
    train_regime_xgb,
)
from notebook_utils._data_explore_functions import line_chart_grid

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)
pd.options.display.float_format = '{:,.5f}'.format

In [ ]:
config = RunConfig(
    round_name='round_1',
    long_short=-1,
    debug_rows=None,
    use_existing_intermediate=True,
    save_intermediate=True,
    save_model=True,
    save_tables=True,
    save_figures=False,
    overwrite_outputs=True,
    use_selected_features=True,
    classification_threshold=0.35,
)

static_opt_config = StaticOptimizationConfig(
    long_short=config.long_short,
    entry_fee=config.entry_fee,
    save_best_json=False,
    save_equity_curve=False,
)

config, static_opt_config

In [ ]:
RUN_DATASET_STAGE = False
RUN_TRAIN_STAGE = False
RUN_STATIC_OPT_STAGE = True
RUN_FULL_PIPELINE = False

In [ ]:
dataset_result = None
training_result = None
full_result = None
static_opt_result = None

if RUN_FULL_PIPELINE:
    full_result = run_full_pipeline(config)
    dataset_result = full_result['dataset']
    training_result = full_result['training']
else:
    if RUN_DATASET_STAGE:
        dataset_result = build_feature_dataset(config)
    if RUN_TRAIN_STAGE:
        training_result = train_regime_xgb(config, dataset=dataset_result)

if RUN_STATIC_OPT_STAGE:
    static_opt_result = run_static_optimization_stage(
        config,
        static_config=static_opt_config,
        training=training_result,
    )

{
    'dataset_loaded': dataset_result is not None,
    'training_run': training_result is not None,
    'full_pipeline_run': full_result is not None,
    'static_opt_run': static_opt_result is not None,
}

In [ ]:
if dataset_result is not None:
    print('Dataset stage source:', dataset_result.source)
    print('Row counts:', dataset_result.row_counts)
    print('Date ranges:', dataset_result.date_ranges)

if training_result is not None:
    print('Best params:', training_result.best_params)
    print('Selected feature count:', len(training_result.selected_feature_columns))
    print('RFECV feature count:', len(training_result.rfecv_feature_columns))
    print('Training artifact paths:')
    for name, path in training_result.artifact_paths.items():
        print(f'  {name}: {path}')

if static_opt_result is not None:
    print('Static optimization artifact paths:')
    for method_name, method_result in static_opt_result.method_results.items():
        print(f' {method_name}: {method_result.candidate_csv_path}')
        if method_result.best_json_path is not None:
            print(f' best_json: {method_result.best_json_path}')
    print(' validation_summary:', static_opt_result.validation_summary_path)

In [ ]:
if static_opt_result is not None:
    print('Selected candidate summaries:')
    display(static_opt_result.selected_candidate_summaries)
    print('')

    for method_name, method_result in static_opt_result.method_results.items():
        print(method_name)
        display(method_result.ranked_candidates.head(3))
        print('')

    print('Validation summary preview:')
    display(static_opt_result.validation_summary.head(3))

In [ ]:
if static_opt_result is not None:
    for rule_name, review in static_opt_result.selected_rule_reviews.items():
        if review.combined_dashboard_path is not None:
            print(f'{rule_name} dashboard: {review.combined_dashboard_path}')
            display(review.combined_dashboard_df.head(10))
            print('')

        print(f'# Grid Equity Curves: {rule_name}')
        fig, axs = plt.subplots(1, 3, figsize=(20, 5))
        datasets = [review.bydate_map.get(partition_name, pd.DataFrame()) for partition_name in static_opt_result.partitions]
        titles = [
            f'{rule_name}: 80% Optmz',
            f'{rule_name}: 20% Optmz',
            f'{rule_name}: OOS Optmz',
        ]

        for ax, data, title in zip(axs, datasets, titles):
            if data is None or data.empty:
                ax.set_title(f'{title} (no data)')
                ax.set_axis_off()
                continue
            line_chart_grid(ax, data, 'normed_date', 'cum_pl_g', 'cum_pl_n', '$ eqt', title)

        plt.tight_layout()
        plt.show()